# Titanic Survival - Fixed Version

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")

## 2. Feature Engineering

In [ ]:
def get_title(name):
    if pd.isna(name): return 'Rare'
    return name.split(', ')[1].split('.')[0]

def prepare(df, is_train=True):
    d = df.copy()
    d['Title'] = d['Name'].apply(get_title)
    tm = {'Mr':1,'Miss':2,'Mrs':3,'Master':4,'Dr':5,'Rev':5,'Col':5,'Major':5,'Capt':5,'Mlle':2,'Ms':2,'Mme':3}
    d['Title'] = d['Title'].map(tm).fillna(5)
    d['FamilySize'] = d['SibSp'] + d['Parch'] + 1
    d['IsAlone'] = (d['FamilySize'] == 1).astype(int)
    d['HasCabin'] = d['Cabin'].notna().astype(int)
    d['Deck'] = d['Cabin'].fillna('U').str[0]
    d['Age'] = d['Age'].fillna(d.groupby(['Pclass','Sex'])['Age'].transform('median')).fillna(d['Age'].median())
    d['Fare'] = d['Fare'].fillna(d.groupby('Pclass')['Fare'].transform('median')).fillna(d['Fare'].median())
    d['Embarked'] = d['Embarked'].fillna(d['Embarked'].mode()[0])
    d['Sex_enc'] = (d['Sex'] == 'male').astype(int)
    d['Embarked_enc'] = LabelEncoder().fit_transform(d['Embarked'])
    d['Deck_enc'] = LabelEncoder().fit_transform(d['Deck'])
    d['IsChild'] = (d['Age'] < 16).astype(int)
    d['IsWomanOrChild'] = ((d['Sex'] == 'female') | (d['Age'] < 16)).astype(int)
    d['ClassSex'] = d['Pclass'] * 10 + d['Sex_enc']
    feats = ['Pclass','Sex_enc','Age','SibSp','Parch','Fare','Embarked_enc','Title','FamilySize','IsAlone','HasCabin','Deck_enc','IsChild','IsWomanOrChild','ClassSex']
    X = d[feats].astype(float)
    if is_train: return X, d['Survived']
    return X

## 3. Prepare

In [ ]:
X_train, y_train = prepare(train)
X_test = prepare(test, is_train=False)
test_id = test['PassengerId']
print(f"Features: {X_train.shape[1]}")

## 4. Train

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gb = GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42)
et = ExtraTreesClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
lr = LogisticRegression(max_iter=1000, random_state=42)
print(f"GB: {cross_val_score(gb, X_train, y_train, cv=cv).mean()*100:.2f}%")
print(f"ET: {cross_val_score(et, X_train, y_train, cv=cv).mean()*100:.2f}%")
print(f"LR: {cross_val_score(lr, X_train, y_train, cv=cv).mean()*100:.2f}%")

## 5. Predict

In [ ]:
gb.fit(X_train, y_train)
et.fit(X_train, y_train)
lr.fit(X_train, y_train)
blend = 0.5*gb.predict_proba(X_test)[:,1] + 0.3*et.predict_proba(X_test)[:,1] + 0.2*lr.predict_proba(X_test)[:,1]
preds = (blend >= 0.5).astype(int)
print(f"Survived: {preds.sum()}")

## 6. Save

In [ ]:
pd.DataFrame({'PassengerId': test_id, 'Survived': preds}).to_csv('submission.csv', index=False)
print("Saved: submission.csv")